# diffBloch Event Report

Renders figures from the canonical `ReportLogger` JSONL event stream. The refinement library writes
no images; plotting and optional figure export happen here.

The plotting code lives in `tools/event_report/figures.py` and the loading code in
`tools/event_report/reader.py` — importable modules with test coverage, not notebook cells. This
notebook is the viewer over them.

Point it at a report by launching Jupyter with `DIFFBLOCH_EVENT_LOG=/path/to/report.jsonl`, editing
the path box below, or dropping a `.jsonl` file on the upload control.


In [ ]:
# Setup: load the report and show the selection controls.
import sys
from pathlib import Path

from IPython.display import Markdown, display

try:
    import ipywidgets as widgets
except ModuleNotFoundError:
    widgets = None

try:
    from tools.event_report import figures, reader
except ModuleNotFoundError:
    # Launched from somewhere other than the checkout root: find it and retry.
    for _candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (_candidate / "pyproject.toml").is_file() and (
            _candidate / "src" / "diffBloch"
        ).is_dir():
            sys.path.insert(0, str(_candidate))
            break
    from tools.event_report import figures, reader

EVENT_LOG = reader.default_event_log()
EXPORT_DIR = Path("event_report_figures")
EXPORT_FORMATS = ("svg",)

event_log_text = None
event_log_upload = None
if widgets is not None:
    event_log_text = widgets.Text(
        value=str(EVENT_LOG), description="JSONL", layout=widgets.Layout(width="80%")
    )
    event_log_upload = widgets.FileUpload(
        accept=".jsonl,application/jsonl,application/x-ndjson",
        multiple=False,
        description="Upload JSONL",
    )
    display(event_log_text, event_log_upload)
else:
    print(f"Using EVENT_LOG={EVENT_LOG}. Install ipywidgets for the path/upload controls.")


def selected_records():
    """The records the controls currently point at; an upload wins over the path box."""
    upload = event_log_upload.value if event_log_upload is not None else None
    if upload:
        item = next(iter(upload.values())) if isinstance(upload, dict) else upload[0]
        return reader.read_records_text(bytes(item["content"]).decode("utf-8"))
    path = Path(event_log_text.value) if event_log_text is not None else EVENT_LOG
    return reader.read_records(path)

In [ ]:
# Re-run this cell after changing the path box or uploading a file.
records = selected_records()
sections = figures.build_sections(records)
print(f"{len(records)} records -> {sum(len(f) for _, f in sections)} figures")

for title, panels in sections:
    display(Markdown(f"## {title}"))
    for figure in panels.values():
        display(figure)

built = {name: figure for _, panels in sections for name, figure in panels.items()}

In [ ]:
# Figure export is opt-in and lives only here, never in the refinement library.
EXPORT_FIGURES = False
figures.export_figures(built, EXPORT_DIR, EXPORT_FORMATS) if EXPORT_FIGURES else []